Problem Statement : Voice Assistant using Python

The objective of this project is to develop a Voice Assistant using Python that can interact with users through voice commands and perform basic tasks automatically. The system should be able to recognize spoken input from the user, convert it into text, process the command, and respond with appropriate actions or voice output.

In [31]:
!pip install SpeechRecognition
!pip install gTTS
!pip install pydub

In [38]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode
import io
from pydub import AudioSegment

def record_audio(filename='recorded.wav', seconds=5):

    js = Javascript('''
    async function recordAudio(seconds){
      const stream = await navigator.mediaDevices.getUserMedia({audio:true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);

      recorder.start();

      await new Promise(resolve => setTimeout(resolve, seconds*1000));

      recorder.stop();

      await new Promise(resolve => recorder.onstop = resolve);

      const blob = new Blob(chunks);
      const buffer = await blob.arrayBuffer();
      const base64 = btoa(
        new Uint8Array(buffer).reduce((data, byte)=> data + String.fromCharCode(byte), '')
      );

      return base64;
    }
    ''')

    display(js)
    audio = output.eval_js(f"recordAudio({seconds})")

    audio_bytes = b64decode(audio)

    with open(filename, 'wb') as f:
        f.write(audio_bytes)

    return filename

In [39]:
import speech_recognition as sr
from pydub import AudioSegment

def speech_to_text(audio_file):

    # Convert the audio file to a standard WAV format using pydub
    try:
        audio = AudioSegment.from_file(audio_file)
        converted_audio_file = "converted_audio.wav"
        audio.export(converted_audio_file, format="wav")
    except Exception as e:
        print(f"Error converting audio file: {e}")
        return ""

    recognizer = sr.Recognizer()

    with sr.AudioFile(converted_audio_file) as source:
        audio = recognizer.record(source)

    try:
        text = recognizer.recognize_google(audio)
        print("You said:", text)
        return text.lower()

    except sr.UnknownValueError:
        print("Could not understand audio")
        return ""
    except sr.RequestError as e:
        print(f"Could not request results from Google Speech Recognition service; {e}")
        return ""

In [40]:
from gtts import gTTS
from IPython.display import Audio
import datetime
import webbrowser

def speak(text):
    tts = gTTS(text)
    tts.save("response.mp3")
    print("Assistant:", text)
    return Audio("response.mp3", autoplay=True)

In [41]:
def process_command(command):

    if "hello" in command:
        return speak("Hello! How can I help you?")

    elif "time" in command:
        time = datetime.datetime.now().strftime("%H:%M")
        return speak(f"The time is {time}")

    elif "date" in command:
        date = datetime.datetime.now().strftime("%d %B %Y")
        return speak(f"Today's date is {date}")

    elif "search" in command:
        return speak("Please search manually in browser because Colab cannot open websites.")

    else:
        return speak("Sorry, I did not understand the command.")

 Output:

In [48]:
audio_file = record_audio()

command = speech_to_text(audio_file)

process_command(command)

<IPython.core.display.Javascript object>

You said: hello
Assistant: Hello! How can I help you?
